# 01 — Raw OSM Dataset Inspection

This notebook performs the initial inspection of the raw OpenStreetMap dataset
used by the ColMaps data pipeline.

The objective is to understand the structure and contents of the source dataset
before defining any filtering or transformation rules. At this stage, the raw
`.osm.pbf` file is inspected without converting its objects into GeoDataFrames
or other in-memory geospatial representations.

Osmium Tool is used for the initial inspection because it can operate directly
on the native OSM PBF representation. Pyrosm, PyArrow, and GeoPandas are
introduced in later processing stages after the raw dataset has been reduced
to the information relevant to ColMaps.

## 1. Input and environment validation

The first step verifies that the expected GeoFabrik extract is available from
the notebook's relative path.

Using a relative path keeps the notebook independent from the absolute
filesystem location of the project.

In [1]:
from pathlib import Path
import subprocess

PBF_PATH = Path("../raw/colombia-latest.osm.pbf")

if not PBF_PATH.exists():
    raise FileNotFoundError(
        f"OSM dataset not found: {PBF_PATH.resolve()}"
    )

print("Dataset:", PBF_PATH.name)
print("Path:", PBF_PATH.resolve())
print(f"Size: {PBF_PATH.stat().st_size / (1024 ** 2):.2f} MB")

Dataset: colombia-latest.osm.pbf
Path: C:\Users\lucas\OneDrive\Documents\ESTUDIOS\Thesis\ColMaps\data-pipeline\raw\colombia-latest.osm.pbf
Size: 311.56 MB


### 1.1 Osmium Tool availability

Raw OSM inspection and coarse preprocessing are performed with Osmium Tool.

Before accessing the dataset, the notebook verifies that the `osmium`
executable is available in the active Conda environment and reports the
installed version.

Recording the tool version is useful for reproducibility because the behaviour
of the preprocessing pipeline can be associated with the software environment
used during execution.

In [2]:
result = subprocess.run(
    ["osmium", "--version"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

print(result.stdout)

osmium version 1.19.0
libosmium version 2.21.0
Supported PBF compression types: none zlib lz4

Copyright (C) 2013-2026  Jochen Topf <jochen@topf.org>
License: GNU GENERAL PUBLIC LICENSE Version 3 <https://gnu.org/licenses/gpl.html>.
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.



## 2. Inspect the raw OSM dataset

Before defining any filtering rules, the raw GeoFabrik extract is inspected
using Osmium Tool.

The objective of this step is to understand the basic characteristics of the
dataset without converting OSM objects into GeoDataFrames or constructing
geometries in memory.

`osmium fileinfo --extended` performs a complete pass over the input file and
reports information such as:

- file format and size;
- declared and calculated bounding boxes;
- dataset timestamps;
- number of nodes, ways, and relations;
- ordering of OSM objects;
- available OSM metadata.

This inspection is intentionally performed with Osmium rather than Pyrosm
because the input is still the complete raw country-level `.osm.pbf` dataset.

In [3]:
result = subprocess.run(
    [
        "osmium",
        "fileinfo",
        "--extended",
        "--no-crc",
        str(PBF_PATH),
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

print(result.stdout)

File:
  Name: ..\raw\colombia-latest.osm.pbf
  Format: PBF
  Compression: none
  Size: 326697976
Header:
  Bounding boxes:
    (-83.23104,-4.25732,-66.8147199,16.5940699)
  With history: no
  Options:
    generator=osmium/1.16.0
    osmosis_replication_base_url=https://download.geofabrik.de/south-america/colombia-updates
    osmosis_replication_sequence_number=4888
    osmosis_replication_timestamp=2026-08-20T20:20:51Z
    pbf_dense_nodes=true
    pbf_optional_feature_0=Sort.Type_then_ID
    sorting=Type_then_ID
    timestamp=2026-08-20T20:20:51Z
Data:
  Bounding box: (-85.9498165,-52.8027348,-64.8146352,26.0037885)
  Timestamps:
    First: 2007-06-29T09:48:27Z
    Last: 2026-08-20T20:19:00Z
  Objects ordered (by type and id): yes
  Multiple versions of same object: no
  CRC32: not calculated (use --crc/-c to enable)
  Number of changesets: 0
  Number of nodes: 47367341
  Number of ways: 5145963
  Number of relations: 36907
  Smallest changeset ID: 0
  Smallest node ID: 4116109
  Small

### Dataset observations

The Colombia GeoFabrik extract contains approximately:

- **47.4 million nodes**
- **5.1 million ways**
- **36.9 thousand relations**

Although the compressed PBF file is approximately **327 MB**, Osmium reports
more than **4.2 GB of decoded buffer data** while scanning the complete file.

This difference illustrates why directly materializing the complete dataset
as Python geospatial objects can require substantially more memory than the
compressed source file size suggests.

The objects are ordered by type and ID, and the dataset does not contain
multiple historical versions of the same OSM object. The latest object
timestamp is `2026-08-20T20:19:00Z`, consistent with the replication timestamp
stored in the file header.

## 3. Discover OSM tag keys present in the dataset

After inspecting the general structure of the dataset, the next step is to
identify the tag keys that actually occur in the Colombia OSM extract.

`osmium tags-count` scans the raw PBF file and counts the occurrence of each
OSM tag key without constructing geometries or loading the dataset into
GeoDataFrames.

Because OpenStreetMap contains a very large and heterogeneous tagging
vocabulary, the complete result may contain a large number of unique keys.
For readability, only the 50 most frequent keys are displayed at this stage,
while the complete output remains available in memory for subsequent analysis.

This step is purely exploratory. No tags are selected or discarded yet.

In [4]:
result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(PBF_PATH),
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

tag_lines = result.stdout.strip().splitlines()

print(f"Unique tag keys found: {len(tag_lines):,}")
print("\nTop 50 most frequent tag keys:\n")

for line in tag_lines[:50]:
    print(line)

Unique tag keys found: 3,213

Top 50 most frequent tag keys:

3467457	"building"
1223593	"highway"
784572	"name"
600596	"source"
598855	"building:levels"
412635	"ref:codcat"
318740	"surface"
237602	"natural"
144222	"oneway"
133432	"lanes"
128796	"amenity"
108972	"landuse"
99349	"addr:city"
99103	"addr:street"
90153	"access"
85750	"waterway"
77916	"place"
71800	"barrier"
68772	"power"
61562	"is_in"
58648	"shop"
58373	"leisure"
52965	"layer"
49654	"divipola"
49497	"addr:state"
47760	"fixme"
47594	"maxspeed"
47024	"addr:housenumber"
46780	"height"
45655	"roof:material"
44904	"crossing"
44800	"alt_name"
40010	"service"
39640	"ref"
37735	"type"
35916	"man_made"
33721	"building:part"
33549	"operator"
30949	"tracktype"
30505	"bridge"
30002	"editor"
29101	"note"
26897	"building:levels:underground"
26656	"catastro:codigo"
26277	"footway"
26141	"operator:type"
24630	"width"
22579	"boundary"
22458	"phone"
21559	"water"


### 3.1 Structure the discovered tag keys

The complete scan identified 3,213 distinct OSM tag keys in the Colombia extract. Frequency alone is not sufficient for determining relevance to ColMaps: a frequently occurring key may be unrelated to tourism, while a less frequent key may represent highly relevant geographic features.

The complete Osmium output is therefore converted into a tabular structure for further exploration. This allows individual keys and groups of keys to be examined without limiting the analysis to the most frequent entries.

In [5]:
import pandas as pd

tag_counts = pd.DataFrame(
    [
        {
            "key": line.split("\t", 1)[1].strip('"'),
            "count": int(line.split("\t", 1)[0]),
        }
        for line in tag_lines
    ]
)

tag_counts

,key,count
0,building,3467457
1,highway,1223593
2,name,784572
3,source,600596
4,building:levels,598855
...,...,...
3208,wikipedia:qu,1
3209,wreck:date_sunk,1
3210,year,1
3211,year_of_founding,1


In [6]:
print(f"Tag keys available for analysis: {len(tag_counts):,}")

Tag keys available for analysis: 3,213


### 3.2 Explore the discovered tag keys

The complete set of 3,213 tag keys is now available as a Pandas DataFrame.

Displaying the entire table is unnecessary and would make the notebook difficult to read. Instead, the dataset is queried selectively to identify keys that may represent geographic feature categories relevant to the ColMaps domain.

The subset is not a final filtering specification. Its purpose is to inspect the actual values occurring in potentially relevant feature families before deciding which OSM classifications should be retained.



In [7]:
tag_counts[
    tag_counts["key"].isin([
        "tourism",
        "historic",
        "natural",
        "leisure",
        "amenity",
        "place",
        "shop",
        "man_made",
        "waterway",
    ])
]

,key,count
7,natural,237602
10,amenity,128796
15,waterway,85750
16,place,77916
20,shop,58648
21,leisure,58373
35,man_made,35916
58,tourism,15225
160,historic,2846


### 3.3 Inspect values within potential feature families

The presence and frequency of an OSM key alone does not determine its relevance to ColMaps.

Each key can contain many values representing substantially different types of geographic objects. A highly frequent family such as `natural=*`, for example, may contain both potentially relevant tourist destinations and features that are outside the intended application scope.

The next step therefore examines the values occurring within potential feature families before deciding whether the family, or specific values within it, should become part of the filtering criteria.

This separates two decisions:

1. identifying OSM keys that may classify relevant geographic features; and
2. selecting the specific `key=value` combinations relevant to ColMaps.

In [8]:
def count_tag_values(tag: str) -> pd.DataFrame:
    result = subprocess.run(
        [
            "osmium",
            "tags-count",
            "--sort=count-desc",
            str(PBF_PATH),
            f"{tag}=*",
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=True,
    )

    rows = []

    for line in result.stdout.strip().splitlines():
        parts = line.split("\t")

        if len(parts) != 3:
            continue

        count, key, value = parts

        rows.append({
            "key": key.strip('"'),
            "value": value.strip('"'),
            "count": int(count),
        })

    return pd.DataFrame(rows)

This function serves as a general wrapper to avoid repeating unnecessary logic that we will print for each interesting tag we have found, avoiding redundancy and maintaining simplicity. It can be used with any tag found in the past stages. If your dataset presents or contains other names that have caught your attention, use this function to explore in depth what these tags contain.

In [9]:
tourism_values = count_tag_values("tourism")

print(f"Tourism occurrences: {tourism_values['count'].sum():,}")
print(f"Distinct Tourism values: {len(tourism_values):,}")

tourism_values

Tourism occurrences: 15,225
Distinct Tourism values: 39


,key,value,count
0,tourism,hotel,5653
1,tourism,hostel,1387
2,tourism,attraction,1191
3,tourism,viewpoint,1040
4,tourism,artwork,1021
5,tourism,information,940
6,tourism,guest_house,883
7,tourism,camp_site,592
8,tourism,alpine_hut,456
9,tourism,apartment,445


#### Tourism tag observations

The `tourism` family contains a mixture of accommodation, attractions, recreational destinations, visitor information, and other tourism-related features.

The most frequent value is `hotel`, followed by `hostel`, `attraction`, `viewpoint`, `artwork`, and `information`. The dataset also contains museums, campsites, huts, zoos, theme parks, galleries, aquariums, and other less frequent tourism features.

The inspection also reveals a small number of unusual or non-standard values, including free text names and combinations of multiple classifications. These values should not automatically be interpreted as valid ColMaps categories and will require validation during the filtering stage.

This confirms that filtering only by the presence of `tourism=*` would be too coarse. Individual values need to be evaluated according to the intended ColMaps feature model.

In [10]:
historic_values = count_tag_values("historic")

print(f"Historic occurrences: {historic_values['count'].sum():,}")
print(f"Distinct Historic values: {len(historic_values):,}")

historic_values

Historic occurrences: 2,846
Distinct Historic values: 55


,key,value,count
0,historic,memorial,754
1,historic,monument,667
2,historic,building,296
3,historic,archaeological_site,251
4,historic,wayside_shrine,157
5,historic,ruins,149
6,historic,yes,109
7,historic,tomb,71
8,historic,boundary_stone,67
9,historic,wayside_cross,67


#### Historic tag observations

The `historic` key contains 55 distinct values in the Colombia extract.

Most occurrences correspond to recognizable historical feature categories, with `memorial`, `monument`, `building`, `archaeological_site`, `wayside_shrine`, and `ruins` representing the most frequent values.

As observed previously with `tourism=*`, the dataset also contains a small number of uncommon or apparently non-standard values. Some values resemble feature names or locally defined classifications rather than reusable OSM categories.

The distribution suggests that `historic` is potentially relevant to the cultural and historical component of ColMaps. However, its individual values should be evaluated before defining the final filtering rules.

In [11]:
natural_values = count_tag_values("natural")

print(f"Natural occurrences: {natural_values['count'].sum():,}")
print(f"Distinct natural values: {len(natural_values):,}")

natural_values.head(15)

Natural occurrences: 237,602
Distinct natural values: 95


,key,value,count
0,natural,tree,108026
1,natural,water,42466
2,natural,wood,39968
3,natural,wetland,8185
4,natural,peak,7159
5,natural,scrub,6622
6,natural,grassland,4659
7,natural,tree_row,4159
8,natural,cliff,3672
9,natural,bare_rock,2654


#### Natural tag observations

The `natural` key is substantially more frequent than the previously inspected
families, with 237,602 occurrences distributed across 95 distinct values.

The distribution is strongly dominated by `tree`, `water`, and `wood`, while
other values occur considerably less frequently.

This distribution demonstrates why selecting all objects matching
`natural=*` would be inappropriate for the ColMaps tourism dataset. The key
describes a broad range of natural geographic features, many of which do not
necessarily represent individual tourist destinations or points of interest.

Consequently, `natural` should be treated as a potential feature family whose
individual values must be evaluated selectively rather than included as a
whole.

In [12]:
leisure_values = count_tag_values("leisure")

print(f"Leisure occurrences: {leisure_values['count'].sum():,}")
print(f"Distinct leisure values: {len(leisure_values):,}")

leisure_values.head(50)

Leisure occurrences: 58,373
Distinct leisure values: 76


,key,value,count
0,leisure,pitch,19912
1,leisure,park,14884
2,leisure,swimming_pool,9500
3,leisure,garden,4798
4,leisure,playground,2634
5,leisure,sports_centre,1760
6,leisure,fitness_centre,729
7,leisure,track,674
8,leisure,fitness_station,525
9,leisure,stadium,412


#### Leisure tag observations

The `leisure` key contains 58,373 occurrences distributed across 76 distinct
values.

The distribution is dominated by `pitch`, `park`, and `swimming_pool`, while
the remaining values represent a broad variety of recreational features,
including gardens, sports centres, stadiums, nature reserves, resorts, and
water parks.

As with `natural=*`, the complete `leisure` family is too broad to be treated
as a tourism-specific classification. Some values may represent relevant
visitor destinations, while others primarily describe local recreational or
sports infrastructure.

Therefore, potential ColMaps features should be selected at the `key=value`
level rather than by retaining all objects tagged with `leisure=*`.

In [13]:
amenity_values = count_tag_values("amenity")

print(f"Amenity occurrences: {amenity_values['count'].sum():,}")
print(f"Distinct amenity values: {len(amenity_values):,}")

amenity_values.head(50)

Amenity occurrences: 128,796
Distinct amenity values: 363


,key,value,count
0,amenity,school,35654
1,amenity,restaurant,14788
2,amenity,parking,10045
3,amenity,place_of_worship,5499
4,amenity,fast_food,4344
5,amenity,fuel,4321
6,amenity,pharmacy,4254
7,amenity,cafe,4136
8,amenity,bank,3323
9,amenity,bench,2768


#### Amenity tag observations

The `amenity` key is highly heterogeneous, with 128,796 occurrences distributed across 363 distinct values.

Unlike more specialized families such as `tourism` or `historic`, `amenity` covers a broad range of facilities and services. The most frequent values include schools, restaurants, parking facilities, places of worship, food establishments, fuel stations, pharmacies, cafés, banks, healthcare facilities, and public services.

Several values may be useful within a tourism-oriented application, including `restaurant`, `cafe`, `bar`, `marketplace`, `ice_cream`, `nightclub`, `pub`, `theatre`, and potentially other cultural, transport, or visitor-support facilities. At the same time, a large proportion of the family describes infrastructure outside the primary tourism scope.

Therefore, retaining `amenity=*` as a whole would introduce a substantial amount of unrelated data. Relevant amenities should instead be selected explicitly at the `key=value` level according to the ColMaps feature model.

In [14]:
place_values = count_tag_values("place")
shop_values = count_tag_values("shop")
man_made_values = count_tag_values("man_made")
waterway_values = count_tag_values("waterway")

In [15]:
family_summary = pd.DataFrame([
    {
        "key": "place",
        "occurrences": place_values["count"].sum(),
        "distinct_values": len(place_values),
    },
    {
        "key": "shop",
        "occurrences": shop_values["count"].sum(),
        "distinct_values": len(shop_values),
    },
    {
        "key": "man_made",
        "occurrences": man_made_values["count"].sum(),
        "distinct_values": len(man_made_values),
    },
    {
        "key": "waterway",
        "occurrences": waterway_values["count"].sum(),
        "distinct_values": len(waterway_values),
    },
])

family_summary

,key,occurrences,distinct_values
0,place,77916,31
1,shop,58648,318
2,man_made,35916,105
3,waterway,85750,28


In [16]:
place_values.head(50)

,key,value,count
0,place,locality,31843
1,place,hamlet,11627
2,place,neighbourhood,9286
3,place,plot,6136
4,place,city_block,3551
5,place,isolated_dwelling,3515
6,place,village,3423
7,place,islet,3347
8,place,farm,2383
9,place,town,1130


#### Place tag observations

The `place` family primarily describes settlements, administrative localities, neighbourhoods, and other named geographic areas rather than individual tourist points of interest.

Values such as `city`, `town`, and `village` may be useful for representing destinations or providing geographic context for tourism features. However, the family also contains highly granular features such as `plot`, `city_block`, `isolated_dwelling`, and `farm`, which are outside the intended scope of tourist POI extraction.

Some geographic values, including `island`, `islet`, and `square`, may have tourism relevance depending on additional attributes or context.

Therefore, `place=*` should not be retained as a general POI selection rule. Selected values may instead support destination modelling, geographic organization, or specific tourism feature categories.

In [23]:
print(f"Shop occurrences: {man_made_values['count'].sum():,}")
print(f"Distinct leisure values: {len(man_made_values):,}")
man_made_values.head(50)

Shop occurrences: 35,916
Distinct leisure values: 105


,key,value,count
0,man_made,petroleum_well,11105
1,man_made,monitoring_station,7703
2,man_made,pipeline,3069
3,man_made,storage_tank,2329
4,man_made,manhole,1390
5,man_made,pier,1294
6,man_made,tower,1087
7,man_made,surveillance,1064
8,man_made,works,995
9,man_made,bridge,542


In [24]:
print(f"Shop occurrences: {shop_values['count'].sum():,}")
print(f"Distinct leisure values: {len(shop_values):,}")
shop_values.head(50)

Shop occurrences: 58,648
Distinct leisure values: 318


,key,value,count
0,shop,convenience,6945
1,shop,supermarket,5589
2,shop,bakery,3953
3,shop,clothes,3015
4,shop,car_repair,2492
5,shop,hairdresser,2342
6,shop,yes,2245
7,shop,hardware,2127
8,shop,car_parts,1408
9,shop,stationery,1384


In [19]:
print(f"Waterway occurrences: {waterway_values['count'].sum():,}")
print(f"Distinct leisure values: {len(waterway_values):,}")
waterway_values.head(30)

Waterway occurrences: 85,750
Distinct leisure values: 28


,key,value,count
0,waterway,stream,64314
1,waterway,river,15331
2,waterway,canal,1805
3,waterway,ditch,1520
4,waterway,drain,1383
5,waterway,waterfall,772
6,waterway,rapids,135
7,waterway,dam,125
8,waterway,dock,107
9,waterway,weir,73


#### Shop tag observations

The `shop` family primarily represents commercial establishments and everyday retail services. Its most frequent values include convenience stores, supermarkets, bakeries, clothing stores, vehicle services, hairdressers, and hardware stores.

Although some values such as `mall`, `travel_agency`, `gift`, or other specialized shops may be useful to visitors, these represent only a subset of a much broader commercial classification.

Therefore, `shop=*` is not considered suitable as a general tourism feature selection criterion. Individual shop categories could be incorporated later if required by the functional scope of ColMaps.

### 3.4 Preliminary findings

The exploratory analysis shows that OSM top-level keys alone are generally too broad to define the ColMaps tourism dataset.

While `tourism` and `historic` contain a high proportion of potentially relevant features, other families such as `natural`, `leisure`, `amenity`, `man_made`, and `shop` combine relevant features with large amounts of data outside the intended tourism scope.

The analysis therefore indicates that the preprocessing strategy should not retain complete OSM feature families indiscriminately. Instead, ColMaps should define its required feature categories first and map those categories to explicit OSM `key=value` combinations.

The `place` family presents a different role: rather than primarily describing individual tourist POIs, it can provide geographic context through settlements and destinations such as cities, towns, and villages.

The next stage therefore defines the geographic feature scope required by ColMaps before constructing the corresponding OSM filtering rules.

## 4. Inspection conclusions

The raw OSM inspection established the main characteristics of the Colombia GeoFabrik extract and provided an initial understanding of its tagging structure.

The dataset contains more than 47 million nodes, 5 million ways, and 3,213 distinct tag keys. Exploratory analysis of several potentially relevant feature families showed that OSM top level keys frequently combine features with very different semantic purposes.

In particular, families such as `natural`, `leisure`, `amenity`, `man_made`, and `shop` contain both potentially useful tourism features and large amounts of unrelated data. Even more tourism-oriented families such as `tourism` and `historic` contain uncommon or non-standard values.

Consequently, the ColMaps preprocessing pipeline will not define its tourism dataset by retaining complete OSM tag families. The required ColMaps feature scope will first be defined and subsequently mapped to explicit OSM `key=value` combinations.

This notebook is limited to exploratory inspection. No transformation or filtering of the source dataset is performed.